### Implement the preprocessing and justify the preprocessing steps

In [3]:
# --- 1. Import required libraries ---
import re
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK resources if you haven't already
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')


import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from tensorflow.keras import layers



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
df = pd.read_csv('CyberBullying Comments Dataset.csv')


# --- Quick look at the dataset ---
print("✅ Data successfully loaded!")
print(f"Number of rows: {len(df)}")
print("\nFirst few rows:")
print(df.head())
print(df.tail())

✅ Data successfully loaded!
Number of rows: 11100

First few rows:
                                                Text  CB_Label
0  damn there is someones nana up here at beach w...         0
1  no kidding! dick clark was a corpse mechanical...         0
2  i read an article on jobros and thought damn w...         0
3  I got one fucking day of sprinkles and now it'...         0
4  I was already listening to Elliott smith  and ...         0
                                                    Text  CB_Label
11095  "Don't worry you little empty head over it ......         1
11096  "Some of Ya'll are dumb as fuck.... These are ...         1
11097  "Lana, you're so full of shit your eyes are br...         1
11098  "You ain't lying let the @dbeeio61:disqus\xa0\...         1
11099  "Looks like that little Cut-n-paste job has go...         1


In [5]:
sample_text = df['Text'].iloc[11097]
print(repr(sample_text))

'"Lana, you\'re so full of shit your eyes are brown. \\xa0\\n\\nNo... they\'re green. \\xa0Like emeralds. \\xa0How did I never see that? \\xa0Lana, your eyes are amazing.I mean, not compared to your tits, but..."'


In [6]:
# Initialize reusable objects
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Function to expand simple contractions
def expand_contractions(text):
    contractions = {
        "can't": "cannot", "won't": "will not", "ain't": "is not", "don't": "do not",
        "'re": " are", "'s": " is", "'d": " would", "'ll": " will",
        "'t": " not", "'ve": " have", "'m": " am"
    }
    for c, expanded in contractions.items():
        text = re.sub(c, expanded, text)
    return text

# Function to clean text
def clean_text(text):
    # 0. Ensure it's a string
    text = str(text)

    # 1. Lowercase
    text = text.lower()

    # 2. Replace URLs and mentions
    text = re.sub(r'http\S+|www\S+|https\S+', ' URL ', text)
    text = re.sub(r'@\w+', ' USER ', text)
    text = re.sub(r'#', '', text)  # remove hashtag symbol, keep word

    # Handle LITERAL escape sequences FIRST (before removing punctuation)
    text = text.replace('\\n', ' ')   # literal \n (two chars)
    text = text.replace('\\r', ' ')   # literal \r (two chars)
    text = text.replace('\\t', ' ')   # literal \t (two chars)

    # Handle ACTUAL escape characters
    text = text.replace('\n', ' ')    # actual newline
    text = text.replace('\r', ' ')    # actual carriage return
    text = text.replace('\t', ' ')    # actual tab
    text = text.replace('\xa0', ' ')  # actual non-breaking space

    # 3. Remove special hidden characters
    text = text.replace('xa0', ' ')   # non-breaking space (do this FIRST)
    # Remove \xc2 AFTER dealing with \xa0
    text = text.replace('xc2', '')

    # 4. Expand contractions
    text = expand_contractions(text)

    # 5. Remove numbers and punctuation (keep ! and ? optionally)
    text = re.sub(r'[0-9]+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation.replace('!', '').replace('?', '')))

    # 6. Normalize repeated characters (e.g., sooooo → sooo)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Tokenization, stopword removal, and lemmatization

def preprocess_spacy(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

def preprocess_text(text):
    text = clean_text(text)
    text = preprocess_spacy(text)
    return text

In [7]:
# Apply preprocessing
df['clean_text'] = df['Text'].apply(preprocess_text)

print(df[['Text', 'clean_text', 'CB_Label']])

                                                    Text  \
0      damn there is someones nana up here at beach w...   
1      no kidding! dick clark was a corpse mechanical...   
2      i read an article on jobros and thought damn w...   
3      I got one fucking day of sprinkles and now it'...   
4      I was already listening to Elliott smith  and ...   
...                                                  ...   
11095  "Don't worry you little empty head over it ......   
11096  "Some of Ya'll are dumb as fuck.... These are ...   
11097  "Lana, you're so full of shit your eyes are br...   
11098  "You ain't lying let the @dbeeio61:disqus\xa0\...   
11099  "Looks like that little Cut-n-paste job has go...   

                                              clean_text  CB_Label  
0      damn someone nana beach not think ic steal qui...         0  
1      kidding ! dick clark corpse mechanically opera...         0  
2      read article jobros think damn cash jobro poke...         0  
3  

In [8]:
print(df.iloc[11095])

Text          "Don't worry you little empty head over it ......
CB_Label                                                      1
clean_text                 worry little head wipe ass hula hoop
Name: 11095, dtype: object


### Extract features and justify the methods used

### Select features and justify the methods used

In [10]:
# Shuffle the dataset randomly before splitting
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['CB_Label'],
    test_size=0.15,
    stratify=df['CB_Label'],
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1111,  # makes final 75/10/15 split
    stratify=y_train,
    random_state=42
)


tfidf = TfidfVectorizer(
    max_features=20000,      # top 20k words
    ngram_range=(1, 2),      # include unigrams + bigrams
    sublinear_tf=True,       # less emphasis on very frequent words
)

X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_val_tfidf   = tfidf.transform(X_val).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# Optional scaling (can slightly help dense layers)
scaler = StandardScaler(with_mean=False)
X_train_tfidf = scaler.fit_transform(X_train_tfidf)
X_val_tfidf   = scaler.transform(X_val_tfidf)
X_test_tfidf  = scaler.transform(X_test_tfidf)



In [11]:
input_dim = X_train_tfidf.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │    10,240,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,306,305 (39.32 MB)

 Trainable params: 10,306,305 (39.32 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', patience=5, mode='max', restore_best_weights=True)
]

history = model.fit(
    X_train_tfidf, y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - auc: 0.5787 - loss: 0.8205 - val_auc: 0.7594 - val_loss: 0.5769
Epoch 2/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.9662 - loss: 0.2499 - val_auc: 0.7441 - val_loss: 0.8153
Epoch 3/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.9952 - loss: 0.0901 - val_auc: 0.7269 - val_loss: 1.0737
Epoch 4/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.9984 - loss: 0.0522 - val_auc: 0.7143 - val_loss: 1.3581
Epoch 5/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.9991 - loss: 0.0376 - val_auc: 0.7089 - val_loss: 1.4907
Epoch 6/50
132/132 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - auc: 0.9994 - loss: 0.0317 - val_auc: 0.7050 - val_loss: 1.5857


In [13]:
y_pred_proba = model.predict(X_test_tfidf).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
              precision    recall  f1-score   support

           0       0.67      0.68      0.68       833
           1       0.68      0.67      0.67       832

    accuracy                           0.68      1665
   macro avg       0.68      0.68      0.68      1665
weighted avg       0.68      0.68      0.68      1665

[[570 263]
 [277 555]]


In [15]:
import joblib

model.save("cyberbullying_nn3.keras")
joblib.dump(tfidf, "tfidf_vectorizer3.pkl")
joblib.dump(scaler, "scaler3.pkl")

['scaler3.pkl']